In [ ]:
import os, sys

# GPU 디바이스
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

# 경로
HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"

os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# logger Jupyter 호환 패치
import logging
import utils.logger as _log_mod

def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.DEBUG)
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S")
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(fmt)
    logger.addHandler(h)
    return logger

_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"작업 디렉토리: {os.getcwd()}")

In [ ]:
!pip install pytorch-forecasting lightning --quiet

In [ ]:
"""
PatchTST 주가 예측 모델 학습
- 시계열을 패치 단위로 분할하여 Transformer에 입력
- 채널 독립 처리로 다변량 시계열에 강함
"""
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.metrics import CrossEntropy
from pytorch_forecasting.data import NaNLabelEncoder

warnings.filterwarnings("ignore")
pl.seed_everything(42)

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {pl.__version__}")

In [ ]:
# ========== 설정 ==========
BASE_PATH = Path(os.environ.get("BASE_PATH", "G:/내 드라이브/kospi200-project"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
MODEL_SAVE_PATH = BASE_PATH / "models" / "patchtst"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

# 학습 설정 (TFT와 동일)
TRAIN_END = "2024-06-30"
VAL_START = "2024-07-01"
VAL_END = "2024-12-31"
TEST_START = "2025-01-01"

# PatchTST 하이퍼파라미터
MAX_ENCODER_LENGTH = 30
MAX_PREDICTION_LENGTH = 1
BATCH_SIZE = 128
MAX_EPOCHS = 50
LEARNING_RATE = 1e-3

# PatchTST specific
PATCH_LENGTH = 5           # 패치 크기 (5거래일 = 1주)
D_MODEL = 64               # 모델 차원
N_HEADS = 4                # 어텐션 헤드 수
N_LAYERS = 2               # 트랜스포머 레이어 수
DROPOUT = 0.3
GRADIENT_CLIP_VAL = 0.1

print("PatchTST 설정 완료")
print(f"  패치 크기: {PATCH_LENGTH}, 모델 차원: {D_MODEL}")
print(f"  헤드: {N_HEADS}, 레이어: {N_LAYERS}")

In [ ]:
# ========== 데이터 로드 ==========
print("피처 로드 중...")
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["sector_id"] = df["sector_id"].astype(str)
df["target_5d"] = df["target_5d"].astype(int)

print(f"  전체: {len(df):,} rows, {df['ticker'].nunique()} tickers")
print(f"  기간: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"  타겟 분포: {df['target_5d'].value_counts().to_dict()}")

In [ ]:
# ========== 시간 기반 분할 ==========
train_df = df[df["date"] <= TRAIN_END].copy()
val_df = df[(df["date"] >= VAL_START) & (df["date"] <= VAL_END)].copy()
test_df = df[df["date"] >= TEST_START].copy()
train_val_df = df[df["date"] <= VAL_END].copy()
train_max_time_idx = train_df["time_idx"].max()

print(f"학습: {len(train_df):,} rows")
print(f"검증: {len(val_df):,} rows")
print(f"테스트: {len(test_df):,} rows")

In [ ]:
# ========== PatchTST 모델 정의 (커스텀 구현) ==========
import torch.nn as nn

class PatchTSTClassifier(pl.LightningModule):
    """PatchTST for binary classification of stock price direction."""
    
    def __init__(
        self,
        n_features: int,
        seq_len: int = 30,
        patch_len: int = 5,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.3,
        n_classes: int = 2,
        lr: float = 1e-3,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        
        # 패치 수 계산
        self.n_patches = seq_len // patch_len
        self.patch_len = patch_len
        
        # 패치 임베딩: (batch, n_features, n_patches, patch_len) -> (batch*n_features, n_patches, d_model)
        self.patch_embedding = nn.Linear(patch_len, d_model)
        
        # 위치 임베딩
        self.pos_embedding = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)
        
        # Transformer 인코더
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # 분류 헤드: 채널 독립 → 평균 풀링 → 분류
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model * n_features, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )
        
        self.loss_fn = nn.CrossEntropyLoss()
    
    def forward(self, x):
        """x: (batch, seq_len, n_features)"""
        batch_size, seq_len, n_feat = x.shape
        
        # (batch, n_feat, seq_len) -> 채널 독립 처리
        x = x.permute(0, 2, 1)
        
        # 패치로 분할: (batch, n_feat, n_patches, patch_len)
        x = x.reshape(batch_size, n_feat, self.n_patches, self.patch_len)
        
        # 채널 독립: (batch * n_feat, n_patches, patch_len)
        x = x.reshape(batch_size * n_feat, self.n_patches, self.patch_len)
        
        # 패치 임베딩
        x = self.patch_embedding(x)  # (batch * n_feat, n_patches, d_model)
        x = x + self.pos_embedding
        
        # Transformer
        x = self.transformer(x)  # (batch * n_feat, n_patches, d_model)
        x = self.norm(x)
        
        # 평균 풀링
        x = x.mean(dim=1)  # (batch * n_feat, d_model)
        
        # 채널 합치기
        x = x.reshape(batch_size, n_feat * self.hparams.d_model)
        
        # 분류
        logits = self.head(x)  # (batch, n_classes)
        return logits
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6
        )
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}}

print("PatchTST 모델 클래스 정의 완료")

In [ ]:
# ========== 데이터셋 준비 (PyTorch Dataset) ==========
from torch.utils.data import Dataset, DataLoader

# 피처 컬럼
observed_features = [
    "daily_return", "log_volume_change", "high_low_range",
    "rsi_14", "macd_norm", "bb_percent_b",
    "volume_ratio_5d", "momentum_5d", "momentum_20d",
    "foreign_net_ratio", "inst_net_ratio",
    "kospi200_return", "vix", "vix_change", "usd_krw_change",
    "relative_return",
]
observed_features = [c for c in observed_features if c in df.columns]
N_FEATURES = len(observed_features)


def make_dataset_with_history(full_df, target_start, target_end, seq_len, features):
    """타겟 기간의 샘플을 만들되, encoder를 위한 과거 데이터를 포함."""
    samples = []
    target_start_ts = pd.Timestamp(target_start)
    target_end_ts = pd.Timestamp(target_end) if target_end else full_df["date"].max()

    for ticker, group in full_df.groupby("ticker"):
        group = group.sort_values("time_idx")
        values = group[features].values.astype(np.float32)
        targets = group["target_5d"].values.astype(np.int64)
        dates = group["date"].values

        for i in range(seq_len, len(group)):
            if dates[i] >= target_start_ts and dates[i] <= target_end_ts:
                x = values[i - seq_len:i]
                y = targets[i]
                samples.append((x, y))
    return samples


class StockSequenceDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x), torch.tensor(y)


# 전체 데이터에서 기간별 샘플 생성
train_samples = make_dataset_with_history(df, "2008-01-01", TRAIN_END, MAX_ENCODER_LENGTH, observed_features)
val_samples = make_dataset_with_history(df, VAL_START, VAL_END, MAX_ENCODER_LENGTH, observed_features)
test_samples = make_dataset_with_history(df, TEST_START, None, MAX_ENCODER_LENGTH, observed_features)

train_dataset = StockSequenceDataset(train_samples)
val_dataset = StockSequenceDataset(val_samples)
test_dataset_pt = StockSequenceDataset(test_samples)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset_pt, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0)

print(f"학습: {len(train_dataset):,} samples")
print(f"검증: {len(val_dataset):,} samples")
print(f"테스트: {len(test_dataset_pt):,} samples")
print(f"피처: {N_FEATURES}개")

In [ ]:
# ========== 모델 초기화 ==========
model = PatchTSTClassifier(
    n_features=N_FEATURES,
    seq_len=MAX_ENCODER_LENGTH,
    patch_len=PATCH_LENGTH,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
    n_classes=2,
    lr=LEARNING_RATE,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"모델 파라미터: {total_params / 1e3:.1f}K")
print(f"패치 수: {model.n_patches} (seq_len={MAX_ENCODER_LENGTH} / patch_len={PATCH_LENGTH})")

In [ ]:
# ========== 학습 ==========
early_stop = EarlyStopping(monitor="val_loss", patience=8, verbose=True, mode="min")
lr_monitor = LearningRateMonitor(logging_interval="epoch")

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    gradient_clip_val=GRADIENT_CLIP_VAL,
    callbacks=[early_stop, lr_monitor],
    enable_progress_bar=True,
    log_every_n_steps=50,
)

print("PatchTST 학습 시작...")
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
print(f"학습 완료! Best val_loss: {early_stop.best_score:.4f}")

In [ ]:
# ========== 성능 평가 ==========
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

best_model = PatchTSTClassifier.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)
best_model.eval()
best_model.cuda()

# 검증 예측
all_probs, all_labels = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.cuda()
        logits = best_model(x)
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y.numpy())

val_probs = np.array(all_probs)
val_actuals = np.array(all_labels)
val_preds = (val_probs >= 0.5).astype(int)

da = accuracy_score(val_actuals, val_preds)
try:
    auc = roc_auc_score(val_actuals, val_probs)
except:
    auc = float("nan")

print("=" * 60)
print("  PatchTST 검증 성능")
print("=" * 60)
print(f"  Direction Accuracy (DA): {da*100:.2f}%")
print(f"  AUC-ROC: {auc:.4f}")
print(f"  양성 예측 비율: {val_preds.mean()*100:.1f}%")
print()
print(classification_report(val_actuals, val_preds, target_names=["하락", "상승"]))
print(f"\n비교: LSTM DA=48.9%, LightGBM DA=50.8%")

In [ ]:
# ========== 테스트 예측 ==========
test_probs_list, test_labels_list = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.cuda()
        logits = best_model(x)
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        test_probs_list.extend(probs)
        test_labels_list.extend(y.numpy())

test_probs = np.array(test_probs_list)
test_actuals = np.array(test_labels_list)
test_preds = (test_probs >= 0.5).astype(int)

test_da = accuracy_score(test_actuals, test_preds)
try:
    test_auc = roc_auc_score(test_actuals, test_probs)
except:
    test_auc = float("nan")

print("=" * 60)
print("  PatchTST 테스트 성능 (OOS)")
print("=" * 60)
print(f"  Direction Accuracy (DA): {test_da*100:.2f}%")
print(f"  AUC-ROC: {test_auc:.4f}")
print()
print(classification_report(test_actuals, test_preds, target_names=["하락", "상승"]))

In [ ]:
# ========== 저장 ==========
import json
from datetime import datetime

trainer.save_checkpoint(str(MODEL_SAVE_PATH / "patchtst_best.ckpt"))

pred_df = pd.DataFrame({"prob": test_probs, "actual": test_actuals, "pred": test_preds})
pred_df.to_parquet(str(MODEL_SAVE_PATH / f"patchtst_predictions_{datetime.now().strftime('%Y%m%d')}.parquet"))

metrics = {
    "model": "PatchTST",
    "val_da": round(da * 100, 2), "val_auc": round(float(auc), 4),
    "test_da": round(test_da * 100, 2), "test_auc": round(float(test_auc), 4),
    "config": {"patch_len": PATCH_LENGTH, "d_model": D_MODEL, "n_heads": N_HEADS, "n_layers": N_LAYERS},
    "timestamp": datetime.now().isoformat(),
}
with open(str(MODEL_SAVE_PATH / "patchtst_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print(f"저장 완료: {MODEL_SAVE_PATH}")

## PatchTST 결과 요약

| 모델 | DA | AUC | 비고 |
|------|-----|-----|------|
| LSTM | 48.9% | - | 시퀀스 20일, 5피처 |
| LightGBM | 50.8% | - | 60+ 피처 |
| TFT | ?% | ? | 30일, 16피처 + static |
| **PatchTST** | **?%** | **?** | 패치 기반 Transformer |

### 특징
- 채널 독립 처리: 각 피처를 독립적으로 Transformer에 입력
- 패치 분할: 5거래일(1주) 단위로 시퀀스를 분할
- 경량: TFT 대비 적은 파라미터